# AI Personal Finance Document Intelligence Agent

## Project Overview

This project demonstrates an AI-powered personal finance assistant
that combines Retrieval-Augmented Generation (RAG), vector search,
Pandas data analysis, and LangGraph agents.

### Key Capabilities

- Process financial documents
- Create document embeddings
- Store embeddings in ChromaDB
- Retrieve relevant financial information
- Analyse transaction data using Pandas
- Use an AI agent to select appropriate tools
- Generate grounded financial responses

### Technology Stack

Python | OpenAI | LangChain | LangGraph | ChromaDB | Pandas


1. Project Setup
2. Import Libraries
3. Load Environment Variables
4. Load Transaction Data
5. Analyse Transactions
6. Load Financial Documents
7. Split Documents
8. Create Embeddings
9. Create ChromaDB
10. Test Document Retrieval
11. Build RAG Function
12. Build Finance Analysis Tool
13. Create LangChain Tools
14. Create LangGraph Agent
15. Test the Agent
16. Project Conclusion

In [1]:
import os
import sys
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Environment ready")

Python version: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
Pandas version: 3.0.5
Environment ready


In [93]:
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OpenAI API key loaded successfully")
else:
    print("OpenAI API key not found")

OpenAI API key loaded successfully


In [94]:
client = OpenAI(api_key=api_key)

print(" OpenAI client created successfully")

 OpenAI client created successfully


In [5]:
response = client.responses.create(
    model="gpt-5.6",
    input="What is RAG? Explain in one sentence."
)

print(response.output_text)

RAG (Retrieval-Augmented Generation) is an AI technique that improves responses by retrieving relevant information from external sources and using it to generate more accurate, context-aware answers.


In [7]:
import sys

print(sys.executable)
print(sys.version)

C:\Users\kvina\AppData\Local\Programs\Python\Python311\python.exe
3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]


In [95]:
import langchain
import langchain_openai
import langgraph
import chromadb
import pandas

print("LangChain:", langchain.__version__)
print("Pandas:", pandas.__version__)

LangChain: 1.4.2
Pandas: 3.0.5


In [10]:
import os

data_folder = "../data"

print("Files in data folder:")
for file in os.listdir(data_folder):
    print(" -", file)

Files in data folder:
 - bank_statement.txt
 - insurance_policy.txt
 - subscriptions.txt
 - transactions.csv
 - utility_bills.txt


In [ ]:
#Reading CSV with Pandas to build the structured-data 

In [12]:
import pandas as pd

csv_path = "../data/transactions.csv"

df = pd.read_csv(csv_path)

print("Number of transactions:", len(df))
df.head()

Number of transactions: 50


,Date,Description,Category,Amount,Type,Month
0,2026-01-02,Salary - ABC Technologies,Income,3200.00,Credit,January
1,2026-01-03,Tesco Supermarket,Groceries,86.45,Debit,January
2,2026-01-04,Netflix,Entertainment,15.99,Debit,January
3,2026-01-05,Amazon,Shopping,42.75,Debit,January
4,2026-01-07,British Gas,Utilities,120.50,Debit,January


In [13]:
print(df.columns)

Index(['Date', 'Description', 'Category', 'Amount', 'Type', 'Month'], dtype='str')


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         50 non-null     str    
 1   Description  50 non-null     str    
 2   Category     50 non-null     str    
 3   Amount       50 non-null     float64
 4   Type         50 non-null     str    
 5   Month        50 non-null     str    
dtypes: float64(1), str(5)
memory usage: 2.5 KB


In [15]:
#Creating first finance analysis
#Total spending

total_spending = df[df["Type"] == "Debit"]["Amount"].sum()

print(f"Total spending: £{total_spending:.2f}")

Total spending: £2849.38


In [16]:
#Total income
total_income = df[df["Type"] == "Credit"]["Amount"].sum()

print(f"Total income: £{total_income:.2f}")

Total income: £10062.50


In [17]:
#Grocery spending
grocery_spending = df[
    (df["Category"] == "Groceries") &
    (df["Type"] == "Debit")
]["Amount"].sum()

print(f"Grocery spending: £{grocery_spending:.2f}")

Grocery spending: £713.15


In [18]:
#Spending by category

category_spending = (
    df[df["Type"] == "Debit"]
    .groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

category_spending

Category
Utilities        722.85
Groceries        713.15
Housing          435.00
Dining           316.45
Shopping         310.79
Transport        162.20
Health           105.00
Entertainment     83.94
Name: Amount, dtype: float64

In [ ]:
#later agent will use this functionality as a tool.

In [ ]:
#Loading the text documents

In [19]:
text_files = [
    "bank_statement.txt",
    "utility_bills.txt",
    "insurance_policy.txt",
    "subscriptions.txt"
]

In [20]:
from langchain_core.documents import Document

documents = []

for filename in text_files:
    file_path = os.path.join("../data", filename)

    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()

    documents.append(
        Document(
            page_content=text,
            metadata={"source": filename}
        )
    )

print("Documents loaded:", len(documents))

Documents loaded: 4


In [21]:
print(documents[0].page_content[:10])

BANK STATE


In [23]:
#Spliting documents into chunks (Instead of sending an entire document to the LLM, i divide it into smaller pieces.)

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))


Number of chunks: 19


In [24]:
print(chunks[0].page_content)
print("\nSource:", chunks[0].metadata["source"])

BANK STATEMENT - SYNTHETIC DATA

Bank Name: DemoBank UK
Account Holder: Alex Morgan
Account Number: XXXX-XXXX-4582
Statement Period: 01 January 2026 - 31 March 2026
Currency: GBP


JANUARY 2026

Source: bank_statement.txt


In [25]:
#Creating OpenAI embeddings (Now converting text into vectors)
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

print("Embedding model ready")

Embedding model ready


In [26]:
#Creating ChromaDB vector database(Now it can store document embeddings.)
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="personal_finance_documents"
)

print("ChromaDB created")

ChromaDB created


In [27]:
#Testing RAG retrieval( Before creating an agent, we must make sure RAG works)

#Create a retriever:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [81]:
for i, chunk in enumerate(chunks):
    if "March 2026" in chunk.page_content and "Netflix" in chunk.page_content:
        print("\n========== CHUNK", i, "==========")
        print(chunk.page_content)




========== CHUNK 16 ==========
SUBSCRIPTION RECORD

Customer Name: Alex Morgan
Customer Reference: DEMO-SUB-45821

------------------------------------------
ACTIVE SUBSCRIPTIONS
------------------------------------------

1. Netflix

Service: Netflix Standard
Provider: Netflix Demo
Category: Entertainment
Monthly Cost: £15.99
Billing Date: 04 of every month
Payment Method: Debit Card
Status: Active

January 2026: £15.99
February 2026: £15.99
March 2026: £15.99


2. Spotify

Service: Spotify Premium
Provider: Spotify Demo
Category: Entertainment
Monthly Cost: £11.99
Billing Date: 12 of every month
Payment Method: Debit Card
Status: Active

January 2026: £11.99
February 2026: £11.99
March 2026: £11.99


3. Gym Membership

========== CHUNK 18 ==========
January 2026: £19.99
February 2026: £19.99
March 2026: £19.99


------------------------------------------
SUBSCRIPTION SUMMARY
------------------------------------------

Netflix: £15.99/month
Spotify: £11.99/month
Fitness Plus: £35.00/

In [31]:
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {len(chunk.page_content)} characters")

Number of chunks: 19
Chunk 0: 292 characters
Chunk 1: 743 characters
Chunk 2: 746 characters
Chunk 3: 167 characters
Chunk 4: 79 characters
Chunk 5: 743 characters
Chunk 6: 746 characters
Chunk 7: 76 characters
Chunk 8: 740 characters
Chunk 9: 746 characters
Chunk 10: 166 characters
Chunk 11: 82 characters
Chunk 12: 775 characters
Chunk 13: 789 characters
Chunk 14: 667 characters
Chunk 15: 374 characters
Chunk 16: 718 characters
Chunk 17: 762 characters
Chunk 18: 791 characters


In [32]:
for i, chunk in enumerate(chunks):
    if "March 2026" in chunk.page_content and "ELECTRICITY" in chunk.page_content:
        print("\n========== CHUNK", i, "==========")
        print(chunk.page_content)


========== CHUNK 12 ==========
UTILITY BILLS - SYNTHETIC CUSTOMER RECORD

Customer Name: Alex Morgan
Customer Reference: DEMO-UK-45821
Statement Period: January 2026 - March 2026


------------------------------------------
ELECTRICITY
------------------------------------------

Provider: Bright Energy UK
Account Number: DEMO-ELEC-10234

January 2026
Billing Date: 15 January 2026
Amount Due: £78.40
Due Date: 25 January 2026
Status: Paid

February 2026
Billing Date: 16 February 2026
Amount Due: £81.30
Due Date: 26 February 2026
Status: Paid

March 2026
Billing Date: 16 March 2026
Amount Due: £84.70
Due Date: 26 March 2026
Status: Paid

Payment Method: Direct Debit


------------------------------------------
GAS
------------------------------------------


In [34]:
#Building a simple RAG answer function (Now i'll connect retrieval to the LLM.)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.4",
    temperature=0,
    api_key=api_key
)

In [37]:
def ask_rag(question):

    results = retriever.invoke(question)

    context = "\n\n".join(
        [
            f"Source: {doc.metadata['source']}\n{doc.page_content}"
            for doc in results
        ]
    )

    prompt = f"""
You are a personal finance document assistant.

Answer the user's question using ONLY the provided context.

If the answer is not present in the context, say:
"I could not find this information in the provided documents."

Context:
{context}

Question:
{question}

Answer clearly and concisely.
"""

    try:
        response = llm.invoke(prompt)
        return response.content

    except Exception as e:
        return f"LLM connection error: {e}"

In [38]:
#Testing
answer = ask_rag(
    "What is my monthly amazon subscription?"
)

print(answer)

I could not find this information in the provided documents.


In [39]:
print(
    ask_rag("What is my COUNCIL TAX bill for January?")
)

Your Council Tax bill for January 2026 is £145.00.


In [64]:
#Created a Pandas finance tool (Here starting  the agent portion)

def analyze_transactions(question): 
    
    question_lower = question.lower() 

    grocery_stores = [
        "tesco",
        "sainsbury",
        "sainsbury's",
        "asda",
        "aldi",
        "lidl",
        "waitrose",
        "morrisons",
        "supermarket"
    ]

    if "grocer" in question_lower or any(store in question_lower for store in grocery_stores): 
        
        grocery_keywords = "|".join(grocery_stores)
        
        result = df[
            (df["Type"] == "Debit") &
            (
                df["Category"].str.contains("Groceries", case=False, na=False) |
                df.astype(str).apply(
                    lambda row: row.str.contains(
                        grocery_keywords, case=False, na=False
                    ).any(),
                    axis=1
                )
            )
        ]["Amount"].sum()

        return f"Total grocery spending: £{result:.2f}" 
 
    if "income" in question_lower: 
        result = df[ 
            df["Type"] == "Credit" 
        ]["Amount"].sum() 
 
        return f"Total income: £{result:.2f}" 
 
    if "spending" in question_lower: 
        result = df[ 
            df["Type"] == "Debit" 
        ]["Amount"].sum() 
 
        return f"Total spending: £{result:.2f}" 
 
    if "category" in question_lower: 
        result = ( 
            df[df["Type"] == "Debit"] 
            .groupby("Category")["Amount"] 
            .sum() 
            .sort_values(ascending=False) 
        ) 
 
        return result.to_string() 
 
    return "I cannot answer this transaction question yet."
    
 
    

In [180]:

def analyze_transactions(question):

    # =========================================================
    # 1. PREPARE DATA
    # =========================================================

    question_lower = question.lower()

    # Make sure Date is datetime
    df["Date"] = pd.to_datetime(df["Date"])

    # =========================================================
    # 2. MONTH MAPPING
    # =========================================================

    months = {
        "january": 1,
        "jan": 1,
        "february": 2,
        "feb": 2,
        "march": 3,
        "mar": 3,
        "april": 4,
        "apr": 4,
        "may": 5,
        "june": 6,
        "jun": 6,
        "july": 7,
        "jul": 7,
        "august": 8,
        "aug": 8,
        "september": 9,
        "sep": 9,
        "sept": 9,
        "october": 10,
        "oct": 10,
        "november": 11,
        "nov": 11,
        "december": 12,
        "dec": 12
    }

    selected_month = None
    selected_month_name = None

    for month_name, month_number in months.items():

        if month_name in question_lower:

            selected_month = month_number

            selected_month_name = pd.Timestamp(
                year=2026,
                month=month_number,
                day=1
            ).strftime("%B")

            break

    # =========================================================
    # 3. CATEGORY ALIASES
    # =========================================================

    category_aliases = {

        "gym": "Gym",
        "fitness": "Gym",

        "entertainment": "Entertainment",

        "travelling": "Travelling",
        "traveling": "Travelling",
        "travel": "Travelling",
        "trvelling": "Travelling",

        "groceries": "Groceries",
        "grocery": "Groceries"
    }

    selected_category = None

    for alias, category in category_aliases.items():

        if alias in question_lower:

            selected_category = category
            break

    # =========================================================
    # 4. MERCHANT ALIASES
    # =========================================================

    merchant_aliases = {

        "mobile phone": "Mobile Phone Bill",
        "mobile": "Mobile Phone Bill",

        "gym membership": "Gym Membership",

        "train travel": "Train Travel",
        "train trvavel": "Train Travel",
        "train travel": "Train Travel",
        "train": "Train Travel"
    }

    selected_merchant = None

    # Check longer aliases first
    for alias in sorted(
        merchant_aliases.keys(),
        key=len,
        reverse=True
    ):

        if alias in question_lower:

            selected_merchant = merchant_aliases[alias]
            break

    # =========================================================
    # 5. FUZZY MERCHANT MATCHING
    # =========================================================

    if selected_merchant is None:

        from difflib import get_close_matches

        merchant_names = (
            df["Description"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        words = question_lower.split()

        for word in words:

            matches = get_close_matches(
                word,
                [
                    merchant.lower()
                    for merchant in merchant_names
                ],
                n=1,
                cutoff=0.75
            )

            if matches:

                matched_lower = matches[0]

                for merchant in merchant_names:

                    if merchant.lower() == matched_lower:

                        selected_merchant = merchant
                        break

            if selected_merchant is not None:
                break

    # =========================================================
    # 6. MONTHLY SPENDING
    # =========================================================

    if (
        "monthly spending" in question_lower
        or "spending every month" in question_lower
        or "spending each month" in question_lower
    ):

        debit_df = df[
            df["Type"] == "Debit"
        ].copy()

        monthly = (
            debit_df
            .groupby(
                debit_df["Date"].dt.to_period("M")
            )["Amount"]
            .sum()
            .sort_index()
        )

        monthly.index = monthly.index.strftime(
            "%B %Y"
        )

        return (
            "Monthly spending:\n\n"
            + monthly.to_string()
        )

    # =========================================================
    # 7. SPENDING BY CATEGORY
    # =========================================================

    if (
        "spending by category" in question_lower
        or "spend by category" in question_lower
        or "category spending" in question_lower
        or "categories" in question_lower
    ):

        category_spending = (
            df[df["Type"] == "Debit"]
            .groupby("Category")["Amount"]
            .sum()
            .sort_values(ascending=False)
        )

        return (
            "Spending by category:\n\n"
            + category_spending.to_string()
        )

    # =========================================================
    # 8. CASH FLOW
    # =========================================================

    if (
        "cash flow" in question_lower
        or "cashflow" in question_lower
    ):

        income = df[
            df["Type"] == "Credit"
        ]["Amount"].sum()

        spending = df[
            df["Type"] == "Debit"
        ]["Amount"].sum()

        cash_flow = income - spending

        return (
            f"Cash Flow Summary:\n"
            f"Total Income: £{income:.2f}\n"
            f"Total Spending: £{spending:.2f}\n"
            f"Net Cash Flow: £{cash_flow:.2f}"
        )

    # =========================================================
    # 9. SPECIFIC CATEGORY
    # =========================================================

    if selected_category is not None:

        matching = df[
            (df["Type"] == "Debit") &
            (
                df["Category"]
                .astype(str)
                .str.lower()
                == selected_category.lower()
            )
        ].copy()

        # Filter by month if user specified one
        if selected_month is not None:

            matching = matching[
                matching["Date"].dt.month
                == selected_month
            ]

        if len(matching) == 0:

            if selected_month_name:

                return (
                    f"No {selected_category} spending "
                    f"found in {selected_month_name}."
                )

            return (
                f"No spending found for "
                f"{selected_category}."
            )

        total = matching["Amount"].sum()

        # Month-wise breakdown
        monthly = (
            matching
            .groupby(
                matching["Date"].dt.to_period("M")
            )["Amount"]
            .sum()
            .sort_index()
        )

        monthly.index = monthly.index.strftime(
            "%B %Y"
        )

        return (
            f"Total spending on {selected_category}: "
            f"£{total:.2f}\n\n"
            f"Monthly breakdown:\n"
            f"{monthly.to_string()}"
        )

    # =========================================================
    # 10. MERCHANT DATE QUERY
    # IMPORTANT: THIS MUST COME BEFORE SPENDING QUERY
    # =========================================================

    if (
        selected_merchant is not None
        and (
            "date" in question_lower
            or "when" in question_lower
            or "transaction date" in question_lower
        )
    ):

        matching = df[
            (df["Type"] == "Debit") &
            (
                df["Description"]
                .astype(str)
                .str.lower()
                .str.contains(
                    selected_merchant.lower(),
                    regex=False,
                    na=False
                )
            )
        ].copy()

        # Filter by month
        if selected_month is not None:

            matching = matching[
                matching["Date"].dt.month
                == selected_month
            ]

        if len(matching) == 0:

            if selected_month_name:

                return (
                    f"No {selected_merchant} transactions "
                    f"found in {selected_month_name}."
                )

            return (
                f"No transactions found for "
                f"{selected_merchant}."
            )

        dates = (
            matching["Date"]
            .sort_values()
            .dt.strftime("%d %B %Y")
            .tolist()
        )

        return (
            f"{selected_merchant} transaction date(s):\n"
            + "\n".join(
                f"- {date}"
                for date in dates
            )
        )

    # =========================================================
    # 11. MERCHANT HIGHEST MONTH
    # =========================================================

    if (
        selected_merchant is not None
        and (
            "which month" in question_lower
            or "what month" in question_lower
            or "highest month" in question_lower
            or "most" in question_lower
        )
    ):

        matching = df[
            (df["Type"] == "Debit") &
            (
                df["Description"]
                .astype(str)
                .str.lower()
                .str.contains(
                    selected_merchant.lower(),
                    regex=False,
                    na=False
                )
            )
        ].copy()

        if len(matching) == 0:

            return (
                f"No transactions found for "
                f"{selected_merchant}."
            )

        monthly = (
            matching
            .groupby(
                matching["Date"].dt.to_period("M")
            )["Amount"]
            .sum()
            .sort_index()
        )

        highest_month = monthly.idxmax()
        highest_amount = monthly.max()

        monthly_display = monthly.copy()

        monthly_display.index = (
            monthly_display.index
            .strftime("%B %Y")
        )

        return (
            f"Highest spending for {selected_merchant} "
            f"was in {highest_month.strftime('%B %Y')}: "
            f"£{highest_amount:.2f}\n\n"
            f"Monthly breakdown:\n"
            f"{monthly_display.to_string()}"
        )

    # =========================================================
    # 12. SPECIFIC MERCHANT SPENDING
    # =========================================================

    if selected_merchant is not None:

        matching = df[
            (df["Type"] == "Debit") &
            (
                df["Description"]
                .astype(str)
                .str.lower()
                .str.contains(
                    selected_merchant.lower(),
                    regex=False,
                    na=False
                )
            )
        ].copy()

        # Filter by month
        if selected_month is not None:

            matching = matching[
                matching["Date"].dt.month
                == selected_month
            ]

        if len(matching) == 0:

            if selected_month_name:

                return (
                    f"No {selected_merchant} transactions "
                    f"found in {selected_month_name}."
                )

            return (
                f"No transactions found for "
                f"{selected_merchant}."
            )

        total = matching["Amount"].sum()

        monthly = (
            matching
            .groupby(
                matching["Date"].dt.to_period("M")
            )["Amount"]
            .sum()
            .sort_index()
        )

        monthly_display = monthly.copy()

        monthly_display.index = (
            monthly_display.index
            .strftime("%B %Y")
        )

        return (
            f"Total spending at {selected_merchant}: "
            f"£{total:.2f}\n\n"
            f"Monthly breakdown:\n"
            f"{monthly_display.to_string()}"
        )

    # =========================================================
    # 13. GROCERY STORES
    # =========================================================

    grocery_stores = [
        "tesco",
        "sainsbury",
        "sainsbury's",
        "asda",
        "aldi",
        "lidl",
        "waitrose",
        "morrisons"
    ]

    for store in grocery_stores:

        if store in question_lower:

            matching = df[
                (df["Type"] == "Debit") &
                (
                    df["Description"]
                    .astype(str)
                    .str.contains(
                        store,
                        case=False,
                        na=False
                    )
                )
            ].copy()

            if len(matching) == 0:

                return (
                    f"No transactions found "
                    f"for {store.title()}."
                )

            total = matching["Amount"].sum()

            breakdown = matching[
                [
                    "Date",
                    "Description",
                    "Category",
                    "Amount"
                ]
            ].to_string(index=False)

            return (
                f"Total spending at "
                f"{store.title()}: £{total:.2f}\n\n"
                f"All matching transactions:\n"
                f"{breakdown}"
            )

    # =========================================================
    # 14. GENERAL GROCERY SPENDING
    # =========================================================

    if (
        "grocery" in question_lower
        or "groceries" in question_lower
    ):

        matching = df[
            (df["Type"] == "Debit") &
            (
                df["Category"]
                .astype(str)
                .str.contains(
                    "Groceries",
                    case=False,
                    na=False
                )
            )
        ]

        total = matching["Amount"].sum()

        return (
            f"Total grocery spending: "
            f"£{total:.2f}"
        )

    # =========================================================
    # 15. INCOME BY SOURCE
    # =========================================================

    if (
        "income" in question_lower
        or "salary" in question_lower
        or "earn" in question_lower
    ):

        income_df = df[
            df["Type"] == "Credit"
        ].copy()

        total_income = income_df["Amount"].sum()

        # Group income by source/description
        income_by_source = (
            income_df
            .groupby("Description")["Amount"]
            .sum()
            .sort_values(ascending=False)
        )

        return (
            f"Total income: £{total_income:.2f}\n\n"
            f"Income by source:\n"
            f"{income_by_source.to_string()}"
        )

    # =========================================================
    # 16. GENERAL TOTAL SPENDING
    # =========================================================

    if (
        "spend" in question_lower
        or "spent" in question_lower
        or "spending" in question_lower
        or "spended" in question_lower
    ):

        result = df[
            df["Type"] == "Debit"
        ]["Amount"].sum()

        return (
            f"Total spending: "
            f"£{result:.2f}"
        )

    # =========================================================
    # 17. GENERAL TOTAL INCOME
    # =========================================================

    if "income" in question_lower:

        result = df[
            df["Type"] == "Credit"
        ]["Amount"].sum()

        return (
            f"Total income: "
            f"£{result:.2f}"
        )

    # =========================================================
    # 18. FALLBACK
    # =========================================================

    return (
        "I cannot answer this transaction question yet."
    )


In [66]:
print(analyze_transactions(
    "How much did I spend on groceries?"
))

Total grocery spending: £713.15


In [67]:
print(analyze_transactions(
    "What is my total spending?"
))

Total spending: £2849.38


In [43]:
#Created tools for the agent(here i'll expose both capabilities as tools)

from langchain_core.tools import tool

In [44]:
#RAG tool -- ChromaDB 

@tool
def search_finance_documents(question: str) -> str:
    """Search personal finance documents and answer document-based questions."""
    
    return ask_rag(question)

In [126]:
#Transaction tool --CSV

@tool
def analyze_finance_transactions(question: str) -> str:
    """Analyze structured financial transactions using Pandas."""
    
    return analyze_transactions(question)

In [135]:
print(search_finance_documents.name)
print(analyze_finance_transactions.name)

search_finance_documents
analyze_finance_transactions


In [152]:

from langgraph.prebuilt import create_react_agent

tools = [
    search_finance_documents,
    analyze_finance_transactions
]

agent = create_react_agent(
    llm,
    tools,
    prompt="""
You are a Personal Finance Assistant.

Use the transaction analysis tool for questions about:
- spending
- transactions
- merchants
- income
- salary
- categories
- dates
- monthly spending

Use the document search tool for questions about:
- bank statement documents
- utility bills
- subscriptions
- insurance policies

Never invent financial information.

If information cannot be found in the provided data,
clearly say that it was not found.
"""
)

print("✅ Finance agent created")


✅ Finance agent created


C:\Users\kvina\AppData\Local\Temp\ipykernel_19572\3774299609.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [129]:
#Testing my AI Agent

#Test 1: RAG

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is my monthly Spotify subscription?"
        }
    ]
})

print(response["messages"][-1].content)

Your Spotify subscription is £11.99 per month.


In [131]:
#Test 2 — Pandas

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much did I spend on groceries at Tesco on 2026-03-03?"
        }
    ]
})

print(response["messages"][-1].content)

£105.30


In [71]:
#Test 2 — Pandas

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much did I spend on groceries?"
        }
    ]
})

print(response["messages"][-1].content)

You spent £713.15 on groceries.


In [153]:
#Test 3 — Insurance document

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What does my insurance policy coverage?"
        }
    ]
})

print(response["messages"][-1].content)

Your insurance policy covers accidental damage to eligible personal property.


In [139]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "how much Coverage limit for my insurance policy?"
        }
    ]
})

print(response["messages"][-1].content)

Your insurance policy coverage limit is £5,000.


In [155]:
#Test 4 — Utility document

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much was my electricity bill in february?"
        }
    ]
})

print(response["messages"][-1].content)



Your electricity bill in February 2026 was **£81.30**.


In [140]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much was my electricity bill in Agust?"
        }
    ]
})

print(response["messages"][-1].content)

I cannot answer this transaction question yet.


In [110]:
#Test 5 —  complex question

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Give me a summary of my financial activity."
        }
    ]
})

print(response["messages"][-1].content)

Here’s a concise summary based on the provided documents:

- Account holder: Alex Morgan
- Bank: DemoBank UK
- Statement period: 01 January 2026 – 31 March 2026
- Payment method for council tax: Direct Debit

Income shown:
- Salary from ABC Technologies: £3,200.00 on 02/01/2026

January 2026 spending shown:
- Groceries: £150.75
- Entertainment: £27.98
- Shopping: £42.75
- Utilities: £120.50
- Housing (Council Tax): £145.00

Council tax:
- Authority: Demo Borough Council
- Account number: DEMO-CT-99821
- Monthly payment: £145.00
- January, February, and March 2026: all marked Paid
- Total for 3 months: £435.00

Utility and household bill summary for 3 months:
- Electricity: £244.40
- Gas: £364.45
- Mobile Phone: £114.00
- Council Tax: £435.00
- Total utility and household bills: £1,157.85

Overall:
- The documents show regular salary income, routine household and living expenses, and council tax paid consistently by direct debit.


In [154]:
print(analyze_transactions("Show me my monthly spending breakdown"))

Monthly spending:

Date
February 2026     938.58
January 2026      877.42
March 2026       1033.38


In [99]:
print(analyze_transactions("Show me my spending by category"))

Spending by category:
Category
Utilities        722.85
Groceries        713.15
Housing          435.00
Dining           316.45
Shopping         310.79
Transport        162.20
Health           105.00
Entertainment     83.94


In [103]:
print(analyze_transactions("How much salary did I get from ABC Technologies, Inc?"))

Total salary from ABC Technologies, Inc.: £9600.00


In [142]:
print(analyze_transactions("How much did I spent at Costa?"))

I cannot answer this transaction question yet.


In [144]:
print(analyze_transactions("How much did I spent in January month?"))

Monthly spending:
Date
February 2026     938.58
January 2026      877.42
March 2026       1033.38


In [125]:
print(analyze_transactions("How much did I spent at Mobile Phone Bill in March?"))

Total spending at Mobile Phone Bill in March: £38.00

Transactions:
      Date       Description  Category  Amount
2026-03-28 Mobile Phone Bill Utilities    38.0


In [166]:
print(analyze_transactions("which month did I spent on  Mobile Phone Bill?"))

Highest spending for Mobile Phone Bill was in January 2026: £38.00

Monthly breakdown:
Date
2026-01    38.0
2026-02    38.0
2026-03    38.0
Freq: M


In [167]:
print(analyze_transactions("How much did I spent on gym?"))

Total spending at Gym Membership: £105.00

Monthly breakdown:
Date
January 2026     35.0
February 2026    35.0
March 2026       35.0


In [168]:
print(analyze_transactions("How much did I spent on Entertainment in March?"))

Total spending on Entertainment in March: £27.98


In [169]:
print(analyze_transactions("How much did I spent on Train travel?"))

Total spending at Train Travel: £162.20

Monthly breakdown:
Date
January 2026     48.6
February 2026    52.4
March 2026       61.2


In [174]:
print(analyze_transactions("Which month did I spent more on Travel ?"))

Highest spending for Train Travel was in March 2026: £61.20

Monthly breakdown:
Date
2026-01    48.6
2026-02    52.4
2026-03    61.2
Freq: M


In [173]:
print(analyze_transactions("How much income i'm getting every month, give me the lst of income?"))

Total income: £10062.50

Income by source:
Description
Salary - ABC Technologies    9600.0
Freelance Payment             450.0
Interest Payment               12.5


In [181]:
print(analyze_transactions("what is the Train Trvavel date in Jan"))

Train Travel transaction date(s):
- 23 January 2026
